## XBRL US API - Python example  
This sample Python code queries assertion data for SEC filings.

### Authenticate for access token 
Click the run button and enter your XBRL US Web account email, account password, Client ID, and secret (get these from https://xbrl.us/access-token), pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [1]:
# @title
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode

api = input('Enter subdomain ("api" or left blank to query public results, otherwise enter a value) ') or 'api'
baseurl = 'https://' + api + '.xbrl.us/'

class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    authurl = baseurl + 'oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}

def refresh(info):
    refresh_auth = {
                'client_id': info.client_id,
                'client_secret' : info.client_secret,
                'grant_type' : 'refresh_token',
                'platform' : 'ipynb',
                'refresh_token' : info.refresh_token
                }
    refreshres = requests.post(info.authurl, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json.get('access_token')
    info.refresh_token = refresh_json.get('refresh_token')
    print('Your access token(%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info

tokenInfo = tokenInfoClass()

# Helper to prompt only if value is missing
def prompt_if_missing(value, prompt_text, secret=False):
    if value:
        return value
    if secret:
        return getpass.getpass(prompt=prompt_text)
    return input(prompt_text)

# Load creds.json (if exists) and ask for user confirmation
creds = {}
use_creds_file = False
if os.path.exists('creds.json'):
    try:
        with open('creds.json', 'r') as f:
            creds = json.load(f)
        confirm = input("Would you like to use credentials on file for this? (yes/no): ").strip().lower()
        use_creds_file = confirm in ('yes', 'y')
        if use_creds_file:
            print("Using credentials on file")
        else:
            print("Enter credentials manually")
            creds = {}
    except Exception as e:
        print("Warning: failed to read creds.json:", e)
        creds = {}

use_test = 'test' in api.lower()

if creds and use_creds_file:
    # Determine credential source: prod > test > top-level > prompts
    selected = None
    
    # First, try nested objects if they exist (e.g., { "prod": {...}} or { "test": {...}})
    if use_test and isinstance(creds.get('test'), dict):
        selected = creds['test']
    elif not use_test and isinstance(creds.get('prod'), dict):
        selected = creds['prod']
    
    # Next, try prefixed keys (prod*/test* if requested)
    if not selected:
        selected = {}
        keys = ['email', 'password', 'client_id', 'client_secret']
        prefix = 'test' if use_test else 'prod'
        for k in keys:
            prefixed_key = prefix + k
            if creds.get(prefixed_key):
                selected[k] = creds.get(prefixed_key)
            # fall back to top-level key if prefix variant not found
            elif creds.get(k):
                selected[k] = creds.get(k)
    
    # Verify we have all required keys
    if not all(selected.get(k) for k in ('email', 'password', 'client_id', 'client_secret')):
        # Fill in missing values from prompts
        selected = {
            'email': selected.get('email'),
            'password': selected.get('password'),
            'client_id': selected.get('client_id'),
            'client_secret': selected.get('client_secret')
        }
    
    # Assign values, prompting for any missing ones
    tokenInfo.email = prompt_if_missing(selected.get('email'), 'Enter your XBRL US Web account email: ')
    tokenInfo.password = prompt_if_missing(selected.get('password'), 'Password: ', secret=True)
    tokenInfo.client_id = prompt_if_missing(selected.get('client_id'), 'Client ID: ', secret=True)
    tokenInfo.client_secret = prompt_if_missing(selected.get('client_secret'), 'Secret: ', secret=True)

    cred_source = 'test credentials' if use_test else 'prod credentials'
    print(f'Using {cred_source}')
else:
    # No creds.json or user declined — prompt the user
    tokenInfo.email = input('Enter your XBRL US Web account email: ')
    tokenInfo.password = getpass.getpass(prompt='Password: ')
    tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
    tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email,
            'client_id': tokenInfo.client_id,
            'client_secret' : tokenInfo.client_secret,
            'password' : tokenInfo.password,
            'grant_type' : 'password',
            'platform' : 'ipynb' }

# Make auth request
payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.authurl, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print("\n\nThere was a problem generating the access token: %s  Run the first cell again and enter the credentials." % (auth_json.get('error_description', auth_json)))
else:
    tokenInfo.access_token = auth_json.get('access_token')
    tokenInfo.refresh_token = auth_json.get('refresh_token')
    if tokenInfo.access_token and tokenInfo.refresh_token:
        print ("\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
    else:
        print("\n\nAuthentication completed but tokens were not returned. Response: {}".format(auth_json))

#print(vars(tokenInfo))
if tokenInfo.access_token and tokenInfo.refresh_token:
    print('\n\naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token)
else:
    print('\n\nNo access token was generated. Check the messages above for errors.')



Your access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. 

For now, skip ahead to the section 'Make a Query'.


access token: 40431ef2-9679-4385-b668-ae4341d2cdf1 refresh token: c4c79079-a492-4482-8b49-aeb4ded90aea


### Make a query 
After the access token confirmation appears above, you can modify the code below to update the query, then run the cell to save it. In the next cell click run to query for results.
  
Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/#/assertion/getAssertionDetails for other endpoints and parameters to filter and return. 

In [2]:
# Define the parameters of the query - this query returns DQC assertions for specified years

endpoint = 'assertion'
XBRL_Elements = [
'DQC.US.0107.9557',
'DQC.US.0117.10093',
'DQC.US.0119.9577',
'DQC.US.0125.9589',
'DQC.US.0139.9857',
'DQC.US.0143.9865',
'DQC.US.0149.9943',
'DQC.US.0157.10077',
'DQC.US.0160.10082',
'DQC.US.0160.10092',
'DQC.US.0165.10091',
'DQC.US.0170.10131',
'DQC.US.0174.10115',
'DQC.US.0174.10117',
'DQC.US.0175.10130',
'DQC.US.0180.10147',
'DQC.US.0180.10148',
'DQC.US.0180.10149',
'DQC.US.0180.10154',
'DQC.US.0184.10164',
'DQC.US.0187.10176',
'DQC.US.0187.10179',
'DQC.US.0188.10183',
'DQC.US.0190.10601',
'DQC.US.0190.10603',
'DQC.US.0190.10604',
'DQC.US.0190.10606',
'DQC.US.0190.10607',
'DQC.US.0190.10612',
'DQC.US.0190.10613',
'DQC.US.0190.10619',
'DQC.US.0192.10620',
'DQC.US.0198.10661',
'DQC.US.0206.10721',
'DQC.US.0207.10726',
'DQC.US.0217.10745',
'DQC.US.0219.10747',
'DQC.US.0225.10790',
'DQC.US.0226.10791',
'DQC.US.0233.10926',
'DQC.US.0233.10927',
'DQC.US.0234.10928',
'DQC.US.0234.10929',
'DQC.US.0235.10931',
'DQC.US.0235.10932',
'DQC.US.0236.10933',
'DQC.US.0237.10934',
'DQC.US.0237.10935',
'DQC.US.0237.10936',
'DQC.US.0241.10942',
'DQC.US.0242.10943',
'DQC.US.0243.10944',
'DQC.US.0243.10945'
    ]
report_year = [
    'us gaap 2026',
    'us gaap 2025'
    ]
fields = [ 
     # this is the list of the characteristics of the data being returned by the query
    'report.base-taxonomy.sort(DESC)',
    'report.filing-date.sort(DESC)',
    'report.entry-url',
    #'report.accepted-timestamp.sort(DESC)',
    'assertion.code.sort(ASC)',
    'report.document-type',
    #'assertion.run-date',
    'report.accession',
    'entity.code',
    'entity.ticker',
    'entity.name',
    'assertion.type',
    #'assertion.detail',
    'assertion.limit()'
    ]

# Set unique rows as True of False (True drops any duplicate rows)
unique = True

# Limit the number of rows displayed by the notebook (does not impact the data frame)
rows_to_display = 2 # Set as '' to display all rows in the notebook

# Below is the list of what's being queried using the search endpoint.
 
params = { 
    'assertion.code': ','.join(XBRL_Elements), 
    'report.base-taxonomy': ','.join(report_year),
    'fields': ','.join(fields)
    }

print('\n\nclick the run button below to execute this query')



click the run button below to execute this query


In [3]:
# @title
# ### Execute the query with loop for all results 
### THIS SECTION DOES NOT NEED TO BE EDITED

search_endpoint = baseurl + 'api/v1/' + endpoint + '/search'
if unique:
    search_endpoint += "?unique"
orig_fields = params['fields']
offset_value = 0
res_df = []
count = 0
query_start = datetime.now()
printed = False
run_query = True

while True:
    if not printed:
        print("On", query_start.strftime("%c"), tokenInfo.email, "(client ID:", str(tokenInfo.client_id.split('-')[0]), "...) started the query and")
        printed = True
    retry = 0
    while retry < 3:
        res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
        res_json = res.json()
        if 'error' in res_json:
            if res_json['error_description'] == 'Bad or expired token':
                tokenInfo = refresh(tokenInfo)
            else: 
                print('There was an error: {}'.format(res_json['error_description']))
                run_query = False
                break
        else: 
		        break
        retry +=1
        if retry >= 3:
            print("Can't refresh the access token.  Run the first query block, then rerun the query.")
            run_query = False

    if not run_query:
       break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else: 
        offset_value += res_json['paging']['limit'] 
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break 
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break 
        params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))
    
    df = pd.DataFrame(res_df)
    # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    pd.options.display.float_format = '{:,.2f}'.format
    display(HTML(df.to_html(max_rows=rows_to_display)))

On Tue Aug 18 13:02:14 2026 david.tauriello@xbrl.us (client ID: 33cccd22 ...) started the query and
up to 5000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 304 records.

At Tue Aug 18 13:03:30 2026, the query finished with   304   rows returned in 0:01:15.970573 for 
https://api.xbrl.us/api/v1/assertion/search?unique&assertion.code=DQC.US.0107.9557,DQC.US.0117.10093,DQC.US.0119.9577,DQC.US.0125.9589,DQC.US.0139.9857,DQC.US.0143.9865,DQC.US.0149.9943,DQC.US.0157.10077,DQC.US.0160.10082,DQC.US.0160.10092,DQC.US.0165.10091,DQC.US.0170.10131,DQC.US.0174.10115,DQC.US.0174.10117,DQC.US.0175.10130,DQC.US.0180.10147,DQC.US.0180.10148,DQC.US.0180.10149,DQC.US.0180.10154,DQC.US.0184.10164,DQC.US.0187.10176,DQC.US.0187.10179,DQC.US.0188.10183,DQC.US.0190.10601,DQC.US.0190.10603,DQC.US.0190.10604,DQC.US.0190.10606,DQC.US.0190.10607,DQC.US.0190.10612,DQC.US.0190.10613,DQC.US.0190.10619,DQC.US.0192.10620,DQC.US.0198.10661,DQC.US.0206.10721,DQC.US.0207.10726,D

,report.base-taxonomy,report.filing-date,report.entry-url,assertion.code,report.document-type,report.accession,entity.code,entity.name,assertion.type
0,US GAAP 2026,2026-08-14,http://www.sec.gov/Archives/edgar/data/1748790/000174879026000022/amcr-20260630.htm,DQC.US.0234.10928,10-K,0001748790-26-000022,0001748790,AMCOR PLC,0234
...,...,...,...,...,...,...,...,...,...
303,US GAAP 2025,2025-04-24,http://www.sec.gov/Archives/edgar/data/1043509/000104350925000014/sah-20250331.htm,DQC.US.0170.10131,10-Q,0001043509-25-000014,0001043509,"SONIC AUTOMOTIVE, INC.",0170


### Define `df` from a local file or uploaded CSV
Use this helper when you want to load `df` from a saved CSV instead of running the XBRL API query.

In [ ]:
import io
from pathlib import Path


def load_df_from_local_file(csv_path):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"Local CSV not found: {csv_path}")
    df_local = pd.read_csv(csv_path)
    print(f"Loaded df from local file: {csv_path} ({len(df_local)} rows)")
    return df_local


def load_df_from_uploaded_csv(uploaded_value):
    if isinstance(uploaded_value, dict):
        uploaded_content = next(iter(uploaded_value.values()))['content']
    else:
        uploaded_content = uploaded_value
    return pd.read_csv(io.BytesIO(uploaded_content))


# Example usage for a local file:
# df = load_df_from_local_file(r"D:\DJT\Documents\GitHub\davidtauriello\dqc_us_rules\tests\input\testfiles\your-file.csv")

# Example usage for notebook upload widgets:
# from ipywidgets import FileUpload
# upload = FileUpload(accept='.csv', multiple=False)
# display(upload)
# df = load_df_from_uploaded_csv(upload.value)

The cell below will create a unique list of assertion.codes for the latest taxonomy version

In [4]:
# prompt: In df return only the first of each assertion.code 

df_unique = df.drop_duplicates(subset=['assertion.code'], keep='first')
df_unique


,report.base-taxonomy,report.filing-date,report.entry-url,assertion.code,report.document-type,report.accession,entity.code,entity.name,assertion.type
0,US GAAP 2026,2026-08-14,http://www.sec.gov/Archives/edgar/data/1748790...,DQC.US.0234.10928,10-K,0001748790-26-000022,0001748790,AMCOR PLC,0234
1,US GAAP 2026,2026-08-14,http://www.sec.gov/Archives/edgar/data/1748790...,DQC.US.0234.10929,10-K,0001748790-26-000022,0001748790,AMCOR PLC,0234
2,US GAAP 2026,2026-08-13,http://www.sec.gov/Archives/edgar/data/2133917...,DQC.US.0170.10131,10-Q,0001731122-26-001056,0002133917,JBAAM Acquisition Corp II,0170
3,US GAAP 2026,2026-08-13,http://www.sec.gov/Archives/edgar/data/1434316...,DQC.US.0237.10934,10-Q,0001193125-26-347988,0001434316,"FATE THERAPEUTICS, INC.",0237
8,US GAAP 2026,2026-08-13,http://www.sec.gov/Archives/edgar/data/1378325...,DQC.US.0237.10935,10-Q,0001378325-26-000036,0001378325,"CAPSOVISION, INC.",0237
15,US GAAP 2026,2026-08-13,http://www.sec.gov/Archives/edgar/data/1769624...,DQC.US.0237.10936,10-Q,0001213900-26-089062,0001769624,Triller Group Inc.,0237
16,US GAAP 2026,2026-08-13,http://www.sec.gov/Archives/edgar/data/88000/0...,DQC.US.0243.10945,10-Q,0001193125-26-349320,0000088000,HORIZON KINETICS HOLDING CORPORATION,0243
23,US GAAP 2026,2026-08-12,http://www.sec.gov/Archives/edgar/data/1534969...,DQC.US.0242.10943,10-Q,0001193125-26-346862,0001534969,"SERA PROGNOSTICS, INC.",0242
36,US GAAP 2026,2026-08-11,http://www.sec.gov/Archives/edgar/data/1738827...,DQC.US.0243.10944,10-Q,0001738827-26-000032,0001738827,"KLX ENERGY SERVICES HOLDINGS, INC.",0243
49,US GAAP 2026,2026-08-10,http://www.sec.gov/Archives/edgar/data/837010/...,DQC.US.0241.10942,10-Q,0000837010-26-000009,0000837010,Voya Retirement Insurance and Annuity Company,0241


In [5]:
# Get unique entity codes from the dataframe and look up tickers using XBRL US API

# Get unique entity codes
unique_entity_codes = df['entity.code'].unique()

# Create a mapping of entity.code to entity.ticker
entity_ticker_map = {}

for entity_code in unique_entity_codes:
    try:
        # Search for the entity using the entity code (CIK)
        url_entity = f"{baseurl}api/v1/entity/search?entity.cik={entity_code}&fields=entity.cik,entity.ticker,entity.name"
        response_entity = requests.get(url_entity, headers={'Authorization': 'Bearer {}'.format(tokenInfo.access_token)})
        if response_entity.status_code == 200:
            entity_data = response_entity.json()
            if entity_data.get('data') and len(entity_data['data']) > 0:
                entity_ticker_map[entity_code] = entity_data['data'][0].get('entity.ticker', '')
            else:
                entity_ticker_map[entity_code] = ''
        elif response_entity.status_code == 401:
            print("Authorization token expired. Try refreshing it.")
            tokenInfo = refresh(tokenInfo)
            break
        else:
            entity_ticker_map[entity_code] = ''
    except Exception as e:
        print(f"Error looking up entity {entity_code}: {e}")
        entity_ticker_map[entity_code] = ''

# Add entity.ticker column between entity.code and entity.name
# First, get the column order
cols = df.columns.tolist()
# Find the position of entity.code
code_idx = cols.index('entity.code')

# Insert entity.ticker at position after entity.code
ticker_col = df['entity.code'].map(entity_ticker_map)
df.insert(code_idx + 1, 'entity.ticker', ticker_col)

print(f"Added entity.ticker for {len(unique_entity_codes)} unique entities")
display(HTML(df.head(10).to_html(max_rows=10)))

Added entity.ticker for 248 unique entities


,report.base-taxonomy,report.filing-date,report.entry-url,assertion.code,report.document-type,report.accession,entity.code,entity.ticker,entity.name,assertion.type
0,US GAAP 2026,2026-08-14,http://www.sec.gov/Archives/edgar/data/1748790/000174879026000022/amcr-20260630.htm,DQC.US.0234.10928,10-K,0001748790-26-000022,0001748790,AUKF/33,AMCOR PLC,0234
1,US GAAP 2026,2026-08-14,http://www.sec.gov/Archives/edgar/data/1748790/000174879026000022/amcr-20260630.htm,DQC.US.0234.10929,10-K,0001748790-26-000022,0001748790,AUKF/33,AMCOR PLC,0234
2,US GAAP 2026,2026-08-13,http://www.sec.gov/Archives/edgar/data/2133917/000173112226001056/e7844_10q.htm,DQC.US.0170.10131,10-Q,0001731122-26-001056,0002133917,JBAAM,JBAAM Acquisition Corp II,0170
3,US GAAP 2026,2026-08-13,http://www.sec.gov/Archives/edgar/data/1434316/000119312526347988/fate-20260630.htm,DQC.US.0237.10934,10-Q,0001193125-26-347988,0001434316,FATE,"FATE THERAPEUTICS, INC.",0237
4,US GAAP 2026,2026-08-13,http://www.sec.gov/Archives/edgar/data/1517396/000162828026056548/ssys-20260630.htm,DQC.US.0237.10934,6-K,0001628280-26-056548,0001517396,SSYS,STRATASYS LTD.,0237
5,US GAAP 2026,2026-08-13,http://www.sec.gov/Archives/edgar/data/1710155/000162828026056493/eye-20260704.htm,DQC.US.0237.10934,10-Q,0001628280-26-056493,0001710155,EYE,"National Vision Holdings, Inc.",0237
6,US GAAP 2026,2026-08-13,http://www.sec.gov/Archives/edgar/data/2003750/000121390026088707/ea0299633-10q_maitong.htm,DQC.US.0237.10934,10-Q,0001213900-26-088707,0002003750,MGSD,"MAITONG SUNSHINE CULTURAL DEVELOPMENT CO., LIMITED",0237
7,US GAAP 2026,2026-08-13,http://www.sec.gov/Archives/edgar/data/769520/000076952026000047/midd-20260704.htm,DQC.US.0237.10934,10-Q,0000769520-26-000047,0000769520,MIDD,THE MIDDLEBY CORPORATION,0237
8,US GAAP 2026,2026-08-13,http://www.sec.gov/Archives/edgar/data/1378325/000137832526000036/cv-20260630.htm,DQC.US.0237.10935,10-Q,0001378325-26-000036,0001378325,CV,"CAPSOVISION, INC.",0237
9,US GAAP 2026,2026-08-13,http://www.sec.gov/Archives/edgar/data/1619856/000161985626000055/crbu-20260630.htm,DQC.US.0237.10935,10-Q,0001619856-26-000055,0001619856,CRBU,"Caribou Biosciences, Inc.",0237


The cell below will save the initial dataframe to a local file or Google Drive as a .csv

In [6]:
# Convert 'report.filing-date' to datetime if not already
df['report.filing-date'] = pd.to_datetime(df['report.filing-date'])

# Group by 'assertion.code', 'report.base-taxonomy', and 'report.document-type', and keep the row with the latest 'report.filing-date'
filtered_df = df.loc[df.groupby(['assertion.code', 'report.base-taxonomy', 'report.document-type'])['report.filing-date'].idxmax()]

# Display the filtered dataframe
display(HTML(filtered_df.to_html(max_rows=rows_to_display)))

print(len(filtered_df))

,report.base-taxonomy,report.filing-date,report.entry-url,assertion.code,report.document-type,report.accession,entity.code,entity.ticker,entity.name,assertion.type
263,US GAAP 2025,2026-02-27,http://www.sec.gov/Archives/edgar/data/1974138/000162828026012576/ncr-20251231.htm,DQC.US.0117.10093,10-K,0001628280-26-012576,0001974138,NATL,NCR ATLEOS CORPORATION,0117
...,...,...,...,...,...,...,...,...,...,...
16,US GAAP 2026,2026-08-13,http://www.sec.gov/Archives/edgar/data/88000/000119312526349320/hkhc-20260630.htm,DQC.US.0243.10945,10-Q,0001193125-26-349320,0000088000,HKHC,HORIZON KINETICS HOLDING CORPORATION,0243


37


In [7]:
# For each (assertion.code, report.document-type) pair, keep only rows with the latest taxonomy year.
# e.g. if a code has entries for US GAAP 2024 and US GAAP 2025, only the 2025 rows are kept.

def taxonomy_year_num(val):
    m = re.search(r'(\d{4})', str(val).strip())
    return int(m.group(1)) if m else 0

# Map each row's base-taxonomy to its numeric year
year_nums = filtered_df['report.base-taxonomy'].map(taxonomy_year_num)

# For each (assertion.code, report.document-type) group, find the max year
max_years = year_nums.groupby([filtered_df['assertion.code'], filtered_df['report.document-type']]).transform('max')

# Keep only rows whose year equals the max for their group
filtered_df = filtered_df[year_nums == max_years]

print(f"Rows after filtering to latest taxonomy year per (assertion.code, document-type): {len(filtered_df)}")
display(HTML(filtered_df.to_html(max_rows=rows_to_display)))

Rows after filtering to latest taxonomy year per (assertion.code, document-type): 29


,report.base-taxonomy,report.filing-date,report.entry-url,assertion.code,report.document-type,report.accession,entity.code,entity.ticker,entity.name,assertion.type
263,US GAAP 2025,2026-02-27,http://www.sec.gov/Archives/edgar/data/1974138/000162828026012576/ncr-20251231.htm,DQC.US.0117.10093,10-K,0001628280-26-012576,0001974138,NATL,NCR ATLEOS CORPORATION,0117
...,...,...,...,...,...,...,...,...,...,...
16,US GAAP 2026,2026-08-13,http://www.sec.gov/Archives/edgar/data/88000/000119312526349320/hkhc-20260630.htm,DQC.US.0243.10945,10-Q,0001193125-26-349320,0000088000,HKHC,HORIZON KINETICS HOLDING CORPORATION,0243


In [8]:
# If you run this program locally, you can save the output to a file 
# on your computer (modify D:\results.csv to your system)

filtered_df.to_csv(r"D:\DJT\Documents\GitHub\davidtauriello\dqc_us_rules\tests\input\testfiles\testcase-tools\missing-dqc-test-v31.csv",sep=",",mode="a",header=False)

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#df.to_csv('assertions-public-exposure.csv')
#!cp data.csv "drive/My Drive/"

The code below filters the dataframe by beginning and ending times for SEC filings, then summarizes the dataframe by consolidating assertion code by major rule and adding a column summarizing the number of unique filings for each assertion code. 

In [10]:
# Convert 'report.accepted-timestamp' to datetime if it's not already
df['report.accepted-timestamp'] = pd.to_datetime(df['report.accepted-timestamp'])

# Define the date range
start_date = '2025-01-01 00:00:00'
end_date = '2025-04-01 00:00:00'

# Filter the dataframe
filtered_df = df[(df['report.accepted-timestamp'] >= start_date) & (df['report.accepted-timestamp'] < end_date)]

# Remove all characters in assertion.code after the third '.' (including the third '.')
filtered_df = filtered_df.map(lambda x: '.'.join(x.split('.')[:3]) if isinstance(x, str) else x)

# Count the occurrences of each assertion.code
assertion_code_counts = filtered_df.groupby('assertion.code').agg(
    count=('assertion.code', 'size'),
    filings=('report.accession', 'nunique')
).reset_index()

# Rename the columns for better readability
assertion_code_counts.columns = ['assertion.code', 'count', 'filings']

# Sort the assertion_code_counts table by 'count' in descending order
assertion_code_counts = assertion_code_counts.sort_values(by='count', ascending=False)

# Display the table
display(HTML(assertion_code_counts.to_html(index=False)))

# Display the updated dataframe
display(HTML(filtered_df.to_html(max_rows=rows_to_display)))

KeyError: 'report.accepted-timestamp'

The cell below will report the creation software used for each assertion in the dataframe.

In [ ]:
for accession in df['report.accession'].unique():
  url_report = f"https://api.xbrl.us/api/v1/report/search?report.accession={accession}&fields=report.accession,report.creation-software"
  response_report = requests.get(url_report, headers={'Authorization': 'Bearer {}'.format(tokenInfo.access_token)})
  accession_output = []
  if response_report.status_code == 200:
    report_data = response_report.json()
    #accession_output += report_data['data'][0]
    print(report_data['data'][0])

  elif response_report.status_code == 401:  # Unauthorized, token might have expired
      print("Authorization token expired. Try refreshing it.")
      tokenInfo = refresh(tokenInfo)  # Call your refresh function
  else:
    print(f"Error fetching data for {accession}: Status code {response_report.status_code}, {response_report.text}")

#print(accession_output)